# Рубежный контроль №1
## Вариант 2
### Тема: Методы обработки данных

**Задача №2** — Кодирование категориального признака методом target (mean) encoding  
**Задача №22** — Масштабирование числового признака по максимальному значению

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MaxAbsScaler

# Загрузка датасета Tips
# Tips — датасет о чаевых в ресторане (244 наблюдения)
# Признаки: total_bill, tip, sex, smoker, day, time, size
df = sns.load_dataset('tips')

print('Датасет Tips:')
print(df.head(10))
print(f'\nРазмер: {df.shape}')
print(f'\nТипы данных:\n{df.dtypes}')
print(f'\nОписательная статистика:\n{df.describe()}')

---
## Задача №2 — Target (Mean) Encoding

**Метод:** для каждой категории признака вычисляется среднее значение целевой переменной (таргета) по всем наблюдениям с данной категорией. Затем исходные категориальные значения заменяются этими средними.

$$X_{encoded}[i] = \frac{1}{|\{j : X[j] = X[i]\}|} \sum_{j : X[j] = X[i]} y[j]$$

**Признак для кодирования:** `day` (день недели) — категориальный  
**Целевая переменная:** `tip` (размер чаевых) — числовая

In [ ]:
feature = 'day'
target = 'tip'

print(f'Признак "{feature}" — распределение по категориям:')
print(df[feature].value_counts())
print(f'\nЦелевая переменная "{target}":')
print(df[target].describe())

In [ ]:
# Target (Mean) Encoding
target_mean = df.groupby(feature)[target].mean()
print('Среднее значение tip для каждого дня (маппинг кодирования):')
print(target_mean)

# Применяем кодирование
df['day_target_encoded'] = df[feature].map(target_mean)

print('\nДатасет после target encoding (первые 15 строк):')
print(df[[feature, target, 'day_target_encoded']].head(15))

In [ ]:
# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Boxplot — распределение tip по дням
order = target_mean.sort_values().index.tolist()
df.boxplot(column=target, by=feature, ax=axes[0], 
           boxprops=dict(color='steelblue'),
           medianprops=dict(color='red', linewidth=2))
axes[0].set_title('Распределение tip по дням')
axes[0].set_xlabel('День недели')
axes[0].set_ylabel('Tip')
plt.sca(axes[0])
plt.title('Распределение tip по дням')

# Barplot — закодированные значения
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
bars = axes[1].bar(target_mean.sort_values().index, 
                   target_mean.sort_values().values,
                   color=colors, edgecolor='black', alpha=0.85)
axes[1].set_title('Target Encoding: среднее значение tip по дням')
axes[1].set_xlabel('День недели')
axes[1].set_ylabel('Среднее значение tip (encoded value)')
for bar, val in zip(bars, target_mean.sort_values().values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.suptitle('')
plt.tight_layout()
plt.show()

print('\nИтоговый маппинг (категория -> закодированное значение):')
print(df[[feature, 'day_target_encoded']].drop_duplicates().sort_values('day_target_encoded').to_string(index=False))

**Вывод по Задаче №2:**  
Методом target (mean) encoding категориальный признак `day` был заменён средним значением чаевых (`tip`) по каждому дню недели:
- Fri → 2.735
- Sat → 2.993  
- Sun → 3.255
- Thur → 2.772

Метод позволяет сохранить информацию о связи категории с целевой переменной и не увеличивает размерность данных (в отличие от one-hot encoding).

> **Важно:** в реальных проектах target encoding применяют с кросс-валидацией или сглаживанием (smoothing), чтобы избежать утечки данных (data leakage).

---
## Задача №22 — Масштабирование по максимальному значению (Max Absolute Scaling)

**Метод:** каждое значение признака делится на максимальное значение по модулю в столбце. Результат находится в диапазоне [-1, 1] (для неотрицательных данных — [0, 1]).

$$X_{scaled} = \frac{X}{\max(|X|)}$$

**Признак для масштабирования:** `total_bill` (сумма счёта)

In [ ]:
feature_scale = 'total_bill'

print(f'Признак "{feature_scale}" до масштабирования:')
print(df[feature_scale].describe())

# Вычисляем максимальное абсолютное значение
max_abs_value = df[feature_scale].abs().max()
print(f'\nМаксимальное абсолютное значение: {max_abs_value}')

In [ ]:
# Применяем масштабирование вручную
df['total_bill_max_scaled'] = df[feature_scale] / max_abs_value

print(f'Признак после масштабирования:')
print(df['total_bill_max_scaled'].describe())

print('\nПервые 15 строк — сравнение исходного и масштабированного признака:')
print(df[[feature_scale, 'total_bill_max_scaled']].head(15))

In [ ]:
# Визуализация: распределение до и после масштабирования
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# До масштабирования
axes[0].hist(df[feature_scale], bins=30, color='coral', edgecolor='black', alpha=0.85)
axes[0].set_title('total_bill — исходные значения')
axes[0].set_xlabel('Значение')
axes[0].set_ylabel('Частота')
axes[0].axvline(df[feature_scale].mean(), color='red', linestyle='--', linewidth=2,
                label=f'Среднее: {df[feature_scale].mean():.2f}')
axes[0].axvline(df[feature_scale].max(), color='darkred', linestyle='-', linewidth=2,
                label=f'Макс: {df[feature_scale].max():.2f}')
axes[0].legend()

# После масштабирования
axes[1].hist(df['total_bill_max_scaled'], bins=30, color='steelblue', edgecolor='black', alpha=0.85)
axes[1].set_title('total_bill — после масштабирования по max')
axes[1].set_xlabel('Значение')
axes[1].set_ylabel('Частота')
axes[1].axvline(df['total_bill_max_scaled'].mean(), color='red', linestyle='--', linewidth=2,
                label=f'Среднее: {df["total_bill_max_scaled"].mean():.4f}')
axes[1].axvline(df['total_bill_max_scaled'].max(), color='darkblue', linestyle='-', linewidth=2,
                label=f'Макс: {df["total_bill_max_scaled"].max():.4f}')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Проверка через sklearn MaxAbsScaler
scaler = MaxAbsScaler()
scaled_sklearn = scaler.fit_transform(df[[feature_scale]])
df['total_bill_sklearn'] = scaled_sklearn

print('Проверка: ручное масштабирование vs sklearn MaxAbsScaler')
print(df[[feature_scale, 'total_bill_max_scaled', 'total_bill_sklearn']].head(10))
match = np.allclose(df['total_bill_max_scaled'], df['total_bill_sklearn'])
print(f'\nРезультаты совпадают: {match} ✓')

print('\nСравнение статистик до и после масштабирования:')
comparison = pd.DataFrame({
    'До масштабирования': df[feature_scale].describe(),
    'После масштабирования': df['total_bill_max_scaled'].describe()
})
print(comparison)

**Вывод по Задаче №22:**  
Масштабирование признака `total_bill` по максимальному значению (Max Absolute Scaling) перевело все значения в диапазон [0, 1]. Форма распределения при этом не изменилась — метод лишь сжимает шкалу, сохраняя все относительные соотношения между наблюдениями. Результат ручного вычисления совпал с результатом `MaxAbsScaler` из библиотеки sklearn.

**Преимущества метода:**
- Прост в реализации
- Не изменяет форму распределения
- Хорошо работает с разреженными данными (sparse matrices)

**Недостатки:**
- Чувствителен к выбросам: если максимальное значение является выбросом, все остальные значения будут сжаты в очень малый диапазон